# 🐟 Huấn Luyện Mô Hình AI Nhận Diện Cá Cảnh Trong Hồ Cá Bằng YOLOv8
### 🎓 Đồ án tốt nghiệp: **Hệ Thống IoT AI Nhận Diện Cá & Giám Sát Môi Trường Hồ Thủy Sinh**
**Sinh viên thực hiện:**
- **Lê Quốc Trí** - MSSV: DH52201621
- **Phạm Minh Thân** - MSSV: DH52201428

**Nền tảng hỗ trợ:** Roboflow Universe & Google Colab (T4/A100 GPU Accelerated)

---
### 📌 Mục tiêu Notebook:
1. Kết nối với **Roboflow** để nạp dataset cá cảnh đã gán nhãn (Object Detection) hoặc phân loại (Classification).
2. Huấn luyện siêu tốc mô hình **YOLOv8** trên phần cứng **GPU T4 miễn phí** của Google Colab.
3. Xuất file trọng số `best.pt` tải về máy để đưa thẳng vào backend Flask `/api/iot/predict` và camera hồ cá (ESP32-CAM / Raspberry Pi).

## ⚡ Bước 1: Kiểm tra cấu hình phần cứng GPU Colab
> Đảm bảo bạn đã bật GPU: Vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print("\n🔥 PyTorch Version:", torch.__version__)
print("🔥 CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("🔥 Tên GPU:", torch.cuda.get_device_name(0))
    print("🔥 Dung lượng VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

## 📦 Bước 2: Cài đặt thư viện (Ultralytics YOLO, Roboflow, OpenCV, Albumentations)

In [ ]:
!pip install -q ultralytics roboflow opencv-python matplotlib seaborn

## 🌐 Bước 3: Nạp Dataset (Chọn Cách 3A qua Roboflow hoặc Cách 3B qua GitHub / Google Drive)

### 🌟 Cách 3A: Nạp dataset từ Roboflow (Khuyên dùng cho Object Detection khoanh vùng cá trong hồ)
Điền **API Key** và tên project trên Roboflow của bạn vào đoạn code bên dưới:

In [ ]:
# Thay thế ROBOFLOW_API_KEY, workspace và project name của bạn
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"  # Nhập API key của bạn từ roboflow.com
WORKSPACE_NAME = "ainhandienca"             # Tên workspace
PROJECT_NAME = "fish-aquarium-detection"     # Tên project
VERSION = 1                                 # Version dataset

if ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
    version = project.version(VERSION)
    dataset = version.download("yolov8")
    print("✅ Đã tải dataset thành công từ Roboflow:", dataset.location)
else:
    print("⚠️ Bạn chưa điền Roboflow API Key. Nếu dùng dataset phân loại 20 loài, hãy chạy Cell Cách 3B bên dưới!")

### 📁 Cách 3B: Tải trực tiếp dataset 20 loài từ GitHub Repository dự án

In [ ]:
import os
if not os.path.exists("ainhandienca"):
    !git clone https://github.com/thanpm080424/ainhandienca.git
%cd ainhandienca
print("Thư mục hiện tại:", os.getcwd())

## 🚀 Bước 4: Huấn luyện mô hình YOLOv8 trên GPU Colab
Hệ thống hỗ trợ 2 tùy chọn huấn luyện tùy theo bài toán của bạn:
- **Tùy chọn 1 (Classification)**: Phân loại 20 loài cá cảnh từ ảnh chụp camera hồ cá.
- **Tùy chọn 2 (Object Detection)**: Phát hiện và khoanh vùng Bounding Box từng con cá đang bơi trong bể kính.

### 🏋️ Tùy chọn 1: Huấn luyện YOLOv8 Classification (20 loài cá)

In [ ]:
# Chạy kịch bản chia tập và fine-tune YOLOv8-cls tối ưu hóa cho môi trường nước
!python train_fish_classifier.py --epochs 50 --batch 32 --device 0 --early-stop 10

### 🎯 Tùy chọn 2: Huấn luyện YOLOv8 Object Detection (Khoanh vùng Bounding Box từ Roboflow)

In [ ]:
# Nếu bạn tải dataset Roboflow Object Detection (có data.yaml):
import os
data_yaml_path = "data.yaml" if os.path.exists("data.yaml") else "../data.yaml"
if os.path.exists(data_yaml_path):
    !yolo detect train data={data_yaml_path} model=yolov8n.pt epochs=50 imgsz=640 batch=16 device=0 optimizer=AdamW
else:
    print("ℹ️ Chưa tìm thấy data.yaml của Roboflow Detection. Hãy tải dataset ở Bước 3A trước khi chạy!")

## 📊 Bước 5: Đánh giá chất lượng mô hình (Đồ thị Loss, Accuracy & Confusion Matrix)

In [ ]:
from PIL import Image
import glob
import matplotlib.pyplot as plt

# Tìm và hiển thị kết quả ma trận nhầm lẫn và biểu đồ huấn luyện
results_imgs = glob.glob("runs/**/results.png", recursive=True) + glob.glob("runs/**/confusion_matrix.png", recursive=True)
for img_path in results_imgs[:2]:
    print("📈 Đang hiển thị:", img_path)
    im = Image.open(img_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(im)
    plt.axis("off")
    plt.show()

## 🔍 Bước 6: Chạy thử nghiệm nhận diện cá trong bể kính (Test Inference)

In [ ]:
from ultralytics import YOLO
import glob

# Tìm checkpoint tốt nhất vừa train
best_weights = glob.glob("runs/**/best.pt", recursive=True)
if best_weights:
    model_path = best_weights[0]
    print("🏆 Nạp trọng số tốt nhất:", model_path)
    model = YOLO(model_path)
    sample_imgs = glob.glob("dataset_ca/*/*.jpg")
    if sample_imgs:
        sample = sample_imgs[0]
        res = model.predict(sample, conf=0.5)[0]
        print("\n🎯 KẾT QUẢ DỰ ĐOÁN:")
        if hasattr(res, "probs") and res.probs is not None:
            top1_id = int(res.probs.top1)
            print(f"- Loài cá: {res.names[top1_id]}")
            print(f"- Độ tin cậy: {float(res.probs.top1conf) * 100:.2f}%")
else:
    print("Chưa có weights best.pt")

## 💾 Bước 7: Tải file trọng số `best.pt` về máy tính
Sau khi tải về, đổi tên hoặc chép file này vào thư mục `ainhandienca/weights/best_fish_yolov8.pt` trên máy tính của bạn.

In [ ]:
from google.colab import files
import shutil
import glob

best_weights = glob.glob("runs/**/best.pt", recursive=True)
if best_weights:
    target_file = best_weights[0]
    shutil.copy(target_file, "best_fish_yolov8.pt")
    print("📥 Đang chuẩn bị tải file best_fish_yolov8.pt về máy...")
    files.download("best_fish_yolov8.pt")
else:
    print("Không tìm thấy file best.pt để tải về.")